In [28]:
import os
import timm
import matplotlib.pyplot as plt
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from tqdm import tqdm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix
from torcheval.metrics import MulticlassF1Score

/home/parasite/.pyenv/versions/3.10.13/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
batch_size = 32
num_epochs = 5
lr = 3e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [30]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [31]:
train_dataset = datasets.ImageFolder(data_dir, transform=transform_train)
val_dataset = datasets.ImageFolder(data_dir, transform=transform_val)

In [32]:
classes = train_dataset.classes
counts = np.bincount([label for _, label in train_dataset.samples])
print("Classes:", classes)
print("Counts:", counts)

Classes: ['Angry', 'Fear', 'Happy', 'Sad', 'Suprise']
Counts: [10148  9732 18439 12553  8227]


In [33]:
class_weights = 1.0 / (counts + 1e-6)
sample_weights = [class_weights[label] for _, label in train_dataset.samples]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


In [34]:
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=len(classes))
model = model.to(device)

In [35]:
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=lr)

In [36]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=False)
    
    for imgs, labels in loop:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}: Train Loss = {avg_loss:.4f}")
    
    model.eval()
    y_true, y_pred = [], []
    f1 = MulticlassF1Score(num_classes=len(classes), average="macro").to(device)
    
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            f1.update(preds, labels)
    
    print(f"Validation F1 (macro): {f1.compute().item():.4f}")

Epoch 1: Train Loss = 1.5490


Validation F1 (macro): 0.2301


Epoch 2: Train Loss = 1.4062


Validation F1 (macro): 0.3140


Epoch 3: Train Loss = 1.3311


Validation F1 (macro): 0.3877


Epoch 4: Train Loss = 1.2758


Validation F1 (macro): 0.4382


Epoch 5: Train Loss = 1.2321


Validation F1 (macro): 0.4453


In [37]:
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=classes, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Classification Report:
              precision    recall  f1-score   support

       Angry     0.3327    0.4781    0.3924     10148
        Fear     0.3459    0.1220    0.1803      9732
       Happy     0.7354    0.6728    0.7027     18439
         Sad     0.4744    0.3083    0.3737     12553
     Suprise     0.4367    0.8524    0.5776      8227

    accuracy                         0.4963     59099
   macro avg     0.4650    0.4867    0.4453     59099
weighted avg     0.5051    0.4963    0.4761     59099

Confusion Matrix:
[[ 4852   614  1419  1390  1873]
 [ 2434  1187  1052  1566  3493]
 [ 2501   380 12406  1157  1995]
 [ 4371   974  1654  3870  1684]
 [  425   277   338   174  7013]]


In [38]:
save_path = "./emotion_classifier.pth"
torch.save(model.state_dict(), save_path)

In [39]:
save_path_whole = "./emotion_classifier_whole.pth"
torch.save(model, save_path_whole)